# State Estimation untuk Digital Twin

Tujuan: membedakan observation dan state, mengimplementasikan prediction/correction, melihat uncertainty saat sensor hilang, dan melakukan fusi dua sensor. Plant ground truth hanya dipakai untuk evaluasi simulator—tidak boleh dibaca estimator.

## Teori

Model skalar: $x_k=x_{k-1}+u_k+w_k$ dan $z_k=x_k+v_k$. Variance proses $Q$ menaikkan uncertainty saat prediction; variance measurement $R$ menentukan kepercayaan pada sensor. Correction memakai innovation $z-x^-$, bukan ground truth.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd() / 'materi/10_digital_twin/src', Path.cwd() / '../10_digital_twin/src']
source_dir = next(path.resolve() for path in candidates if path.exists())
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from digital_twin_lab.estimation import ScalarKalmanFilter, fuse_measurements
from digital_twin_lab.simulation import run_tank_experiment

## Eksperimen 1 — noisy sensor, dropout, dan model mismatch

Fault clog dimulai pada langkah 300. Estimator tetap memakai coefficient normal sehingga residual mengukur mismatch.

In [ ]:
rows = run_tank_experiment(steps=500, seed=42, fault_step=300, dropout_probability=0.08)
result = pd.DataFrame(rows)
result['absolute_error_m'] = (result['estimate_m'] - result['true_level_m']).abs()
print(result[['absolute_error_m', 'variance_m2']].describe())
print('dropout:', result['measurement_m'].isna().sum())
print('alarm pertama:', result.loc[result['alarm'], 'step'].min())

In [ ]:
sigma = np.sqrt(result['variance_m2'])
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(result['step'], result['true_level_m'], label='ground truth simulator')
axes[0].scatter(result['step'], result['measurement_m'], s=5, alpha=0.25, label='observation')
axes[0].plot(result['step'], result['estimate_m'], label='estimated state')
axes[0].fill_between(result['step'], result['estimate_m'] - 1.96*sigma,
                     result['estimate_m'] + 1.96*sigma, alpha=0.2, label='nominal 95% interval')
axes[0].axvline(300, color='red', linestyle='--', label='fault onset')
axes[0].set_ylabel('level (m)')
axes[0].legend()
axes[1].plot(result['step'], result['variance_m2'], label='state variance')
axes[1].scatter(result.loc[result['measurement_m'].isna(), 'step'],
                result.loc[result['measurement_m'].isna(), 'variance_m2'],
                color='orange', s=12, label='sensor missing')
axes[1].set_xlabel('step')
axes[1].set_ylabel('variance (m²)')
axes[1].legend()
plt.tight_layout()

Interval di atas hanya nominal. Uji empirical coverage terhadap ground truth simulator; di dunia nyata, ground truth mungkin memerlukan reference instrument atau inspection. Coverage buruk dapat berarti Q/R/model/distribution salah.

In [ ]:
lower = result['estimate_m'] - 1.96*sigma
upper = result['estimate_m'] + 1.96*sigma
coverage = result['true_level_m'].between(lower, upper).mean()
print(f'empirical coverage: {coverage:.1%}')

## Eksperimen 2 — multi-sensor fusion

Sensor independen dengan variance lebih kecil harus mendapat bobot lebih besar. Jika error sensor berkorelasi, fungsi sederhana ini terlalu percaya diri.

In [ ]:
prior = (2.0, 0.5)
sensor_a = (2.10, 0.01)  # precise
sensor_b = (2.80, 0.25)  # noisy
fused_estimate, fused_variance = fuse_measurements(*prior, [sensor_a, sensor_b])
print({'estimate': fused_estimate, 'variance': fused_variance})
assert abs(fused_estimate - sensor_a[0]) < abs(fused_estimate - sensor_b[0])
assert fused_variance < min(prior[1], sensor_a[1], sensor_b[1])

## Latihan dan checkpoint

1. Bandingkan dropout 0%, 10%, dan 40%: MAE, maximum variance, interval coverage.
2. Tambahkan bias sensor +0.3 m. Mengapa estimator bisa confident tetapi salah?
3. Simulasikan sensor kedua dengan variance berbeda dan dropout independen.
4. Tunda 10% observation selama lima langkah. Jangan memperlakukannya sebagai reading saat ini; pilih drop-late, buffer, atau replay dan jelaskan trade-off.
5. Jelaskan beda process variance Q dan measurement variance R tanpa melihat catatan.
6. Mengapa smoother offline tidak boleh dilaporkan sebagai performa estimator online?